# 两者结合（Megatron-DeepSpeed）通常是目前训练千亿参数以上模型的行业标准。
- <font color='red'>Megatron 负责把模型“切得碎”（算得快），DeepSpeed 负责把显存“挤得干”（省得多），两者联手才能训好现在的千亿参数大模型。</font>

| 特性 | Megatron-LM (NVIDIA) | DeepSpeed (Microsoft) |
| :--- | :--- | :--- |
| 核心强项 | 3D 并行策略 (张量并行 TP + 流水并行 PP + 数据并行 DP) | ZeRO 显存优化 (消除冗余) + 内核融合 |
| 主要解决的问题 | <font color='red'>模型太大，单卡/单节点放不下，必须把模型切开算。</font> | <font color='red'>显存不够用（即使模型能放下，优化器状态也占太多），想省显存或增大 Batch Size。</font> |
| 并行粒度 | <font color='green'>细粒度：在算子级别切分矩阵乘法（TP），在层级别切分模型（PP）。</font> | <font color='red'>粗粒度：主要在数据并行（DP）基础上，对优化器状态、梯度、参数进行分片。</font> |
| 代码侵入性 | 高：通常需要按照 Megatron 的特定格式重写模型代码。 | 低：只需几行代码即可集成到现有 PyTorch 模型中（尤其是 ZeRO）。 |
| 生态背景 | NVIDIA GPU 硬件深度优化，依赖 NCCL。 | 微软 Azure 云环境优化，支持多种硬件后端。 |



最佳实践方案：Megatron-DeepSpeed 混合架构
- <font color='red'>节点内 (Intra-node)</font>：<b>使用 Megatron 的 张量并行 (TP)。</b>利用 NVLink 的高带宽，快速完成算子级切分计算。
- <font color='red'>节点间 (Inter-node)</font>：<b>使用 DeepSpeed 的 ZeRO (通常是 ZeRO-1 或 ZeRO-2) 或 Megatron 的 流水并行 (PP)。</b>

- 具体组合：
    - TP (Megatron) 处理大矩阵计算。
    - PP (Megatron/DeepSpeed) 处理层间切分。
    - DP + ZeRO (DeepSpeed) 处理数据并行时的显存冗余。

| 场景 | 推荐方案 | 理由 |
| :--- | :--- | :--- |
| 中小模型 (<10B) | DeepSpeed (ZeRO-2/3) | 配置简单，无需改写模型代码，显存节省效果立竿见影，单卡/多卡微调首选。 |
| 超大模型 (>100B) 预训练 | Megatron-DeepSpeed | 必须利用 TP/PP 来保证计算效率，同时利用 ZeRO 来优化显存。这是目前工业界的标准。 |
| 推理 (Inference) | DeepSpeed-Inference 或 vLLM | DeepSpeed 提供了专门的推理引擎，支持动态批处理和算子融合；vLLM 则在 PagedAttention 上表现更佳。 |
| 科研/快速验证 | PyTorch FSDP | PyTorch 原生支持的 Fully Sharded Data Parallel，功能类似 DeepSpeed ZeRO-3，无需安装额外库，生态兼容性好。 |



# Megatron-DeepSpeed代码示例

Megatron-DeepSpeed 并非一个独立的库，而是将 NVIDIA 的 Megatron-LM 框架与微软的 DeepSpeed 库结合使用的方案。这种组合旨在训练万亿参数级别的超大模型，<font color='red'>它融合了 Megatron-LM 在模型并行（如张量并行、流水线并行）方面的优势和 DeepSpeed 在显存优化（如 ZeRO 系列优化器）方面的能力。</font>

下面我将通过一个完整的代码示例，详细解释如何构建和配置一个 Megatron-DeepSpeed 训练任务。


## 🏗️ 整体架构

在开始编码前，理解其工作流程至关重要：

1. 模型定义 (Megatron-LM): 使用 Megatron-LM 的 API 定义模型结构。
    - <font color='red'>Megatron-LM 负责将模型的各个层（如 Transformer Block）在多个 GPU 之间进行切分，实现张量并行 (TP) 和 流水线并行 (PP)。</font>
2. 优化与封装 (DeepSpeed): 
    - 使用 DeepSpeed 来初始化优化器、学习率调度器，并应用 ZeRO 优化策略。
    - <font color='red'>DeepSpeed 会将 Megatron-LM 定义的模型再次进行封装，负责数据并行 (DP) 层面的显存优化，例如将优化器状态和梯度进行分片。</font>
3. 协同训练: <font color='red'>训练循环由 DeepSpeed 的引擎驱动，它会自动处理前向传播、反向传播、梯度同步（通过 all_reduce 等）和参数更新等复杂步骤。</font>


## 💻 完整代码示例
以下是一个简化的 GPT 模型训练脚本示例，展示了如何结合两者。

In [ ]:
import torch
import deepspeed
import argparse
# 假设已安装 megatron-lm 包
from megatron import get_args, get_tokenizer
from megatron.training import train_step
from megatron.model import GPTModel
from megatron.optimizer import get_megatron_optimizer
from megatron.initialize import initialize_megatron

def model_provider(pre_process=True, post_process=True):
    """
    使用 Megatron-LM 的 API 构建模型。
    Megatron 负责模型内部的并行化，如张量并行。
    """
    print(f"Rank {torch.distributed.get_rank()}: 正在构建 Megatron 模型...")
    model = GPTModel(
        num_layers=12,
        hidden_size=768,
        num_attention_heads=12,
        pre_process=pre_process,
        post_process=post_process
    )
    return model

def add_model_config_args(parser):
    """添加模型相关的命令行参数"""
    group = parser.add_argument_group(title='model')
    group.add_argument('--num-layers', type=int, default=12)
    group.add_argument('--hidden-size', type=int, default=768)
    group.add_argument('--num-attention-heads', type=int, default=12)
    return parser

def main():
    # 1. 初始化 Megatron 环境
    # 这会设置分布式环境、解析命令行参数等
    initialize_megatron(
        extra_args_provider=add_model_config_args,
        args_defaults={'tokenizer_type': 'GPT2BPETokenizer'}
    )
    
    # 获取所有参数
    args = get_args()

    # 2. 构建模型 (使用 Megatron-LM)
    model = model_provider()

    # 3. 构建优化器 (使用 DeepSpeed)
    # 定义 DeepSpeed 的配置，这是结合的核心
    ds_config = {
        "train_batch_size": args.global_batch_size,
        "gradient_accumulation_steps": args.gradient_accumulation_steps,
        "steps_per_print": 10,
        "optimizer": {
            "type": "Adam",
            "params": {
                "lr": args.lr,
                "weight_decay": 0.01
            }
        },
        "fp16": {
            "enabled": True,
            "loss_scale": 0,
            "loss_scale_window": 1000,
            "initial_scale_power": 16,
            "hysteresis": 2,
            "min_loss_scale": 1
        },
        # 【核心】ZeRO 优化配置
        "zero_optimization": {
            "stage": 2, # 使用 ZeRO-2，对优化器状态和梯度进行分片
            "allgather_partitions": True,
            "allgather_bucket_size": 2e8,
            "reduce_scatter": True,
            "reduce_bucket_size": 2e8,
            "overlap_comm": True,
            "contiguous_gradients": True,
        }
    }

    # 使用 DeepSpeed 初始化模型、优化器和数据加载器
    # 这一步将 Megatron 模型包装在 DeepSpeed 引擎中
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        optimizer=None, # 让 DeepSpeed 创建优化器
        config=ds_config,
        dist_init_required=False # Megatron 已经初始化过分布式环境
    )

    # 4. 训练循环
    # 在实际应用中，这里会从数据加载器获取数据并进行迭代
    print(f"Rank {torch.distributed.get_rank()}: 开始训练...")
    
    # 模拟训练步骤
    for step in range(100):
        # 构造一个虚拟的批次数据
        # 在实际场景中，data 和 labels 来自你的数据集
        data = torch.randint(0, 1000, (args.micro_batch_size, 128)).cuda()
        labels = data.clone()
        
        # 前向传播
        # DeepSpeed 会自动处理并行策略下的前向计算
        output = model_engine(data)
        
        # 计算损失
        loss = torch.nn.functional.cross_entropy(output.view(-1, output.size(-1)), labels.view(-1))
        
        # 反向传播
        # DeepSpeed 会自动处理梯度的反向传播、ZeRO 分片同步等
        model_engine.backward(loss)
        
        # 更新参数
        # DeepSpeed 会自动处理优化器状态的更新和参数同步
        model_engine.step()
        
        if step % 10 == 0:
            print(f"Rank {torch.distributed.get_rank()}: Step {step}, Loss: {loss.item():.4f}")

if __name__ == "__main__":
    main()

## 📝 启动脚本
- 你需要一个 Shell 脚本来通过 torchrun 或 deepspeed 命令启动这个分布式任务。

In [ ]:
#!/bin/bash

# 使用 torchrun 启动，这是 PyTorch 推荐的分布式启动方式
# --nproc_per_node=8 表示使用当前机器上的 8 个 GPU
# --nnodes=1 表示使用 1 台机器
# --node_rank=0 表示当前机器的 rank 为 0

GPUS_PER_NODE=8
NNODES=1
NODE_RANK=0
MASTER_ADDR=localhost
MASTER_PORT=6000

torchrun --standalone \
       --nnodes=$NNODES \
       --nproc_per_node=$GPUS_PER_NODE \
       --node_rank=$NODE_RANK \
       --master_addr=$MASTER_ADDR \
       --master_port=$MASTER_PORT \
       train.py \
       --num-layers 24 \
       --hidden-size 1024 \
       --num-attention-heads 16 \
       --global-batch-size 1024 \
       --micro-batch-size 8 \
       --gradient-accumulation-steps 16 \
       --lr 1e-4

## 🔑 关键配置详解
1. <font color='green'>DeepSpeed 配置 (ds_config)</font>

这是整个方案的核心，它告诉 DeepSpeed 如何优化训练过程。
- <b>optimizer:</b> 定义优化器类型和超参数。DeepSpeed 会自动创建并管理它。
- <b>fp16:</b> 启用混合精度训练，这对于大模型至关重要，可以节省显存并加速计算。
- <b>zero_optimization:</b> ZeRO (Zero Redundancy Optimizer) 是 DeepSpeed 的灵魂。
    - stage:
        1: 仅对优化器状态进行分片。
        2: 对优化器状态和梯度进行分片。这是最常用的配置，在显存节省和通信开销之间取得了良好平衡。
        3: 对优化器状态、梯度和模型参数进行分片。显存占用最低，可以训练更大的模型，但通信开销也最大。
    - overlap_comm: 允许通信和计算重叠，隐藏通信延迟，提升效率。
    
2. <font color='green'>Megatron-LM 并行参数</font>

这些参数通常通过命令行传递，控制模型如何切分。
- --tensor-model-parallel-size (TP): <font color='red'>指定张量并行的 GPU 数量。</font>例如，设为 2 会将每个 Transformer 层的权重矩阵在 2 个 GPU 上切分。
- --pipeline-model-parallel-size (PP): <font color='red'>指定流水线并行的 GPU 数量。</font>例如，设为 4 会将 24 层模型分配到 4 个 GPU 上，每个 GPU 负责 6 层。
- --data-parallel-size (DP): <font color='red'>数据并行的大小。</font>通常由总 GPU 数量、TP 和 PP 的大小自动决定 (DP = 总GPU数 / (TP * PP))。DeepSpeed 的 ZeRO 优化是在 DP 层面工作的。

## 🤝 协同工作流程
1. <font color='green'>模型切分 (Megatron)</font>: 当你设置 TP=2, PP=4 时，Megatron-LM 会在内部创建一个 3D 并行的模型。一个完整的 Transformer 层被切分到 2 个 GPU 上（TP），而 24 个层被分配到 4 个 GPU 上（PP）。

2. <font color='green'>数据并行与 ZeRO (DeepSpeed)</font>: 剩下的 GPU 用于数据并行。对于每个微批次（micro-batch）的数据，DeepSpeed 会将其分发到不同的数据并行组。在反向传播后，DeepSpeed 的 ZeRO-2 优化器会将梯度进行分片，每个 GPU 只保留自己负责的那部分梯度，从而极大地节省了显存。

3. <font color='green'>统一执行 (DeepSpeed Engine)</font>: model_engine.backward(loss) 和 model_engine.step() 这两个调用，背后是 DeepSpeed 和 Megatron-LM 的复杂协同。DeepSpeed 负责调度整个流程，并在关键时刻调用 Megatron-LM 的通信函数来处理 TP 和 PP 带来的特殊同步需求。

通过这种方式，Megatron-DeepSpeed 方案能够高效地利用成千上万个 GPU 来训练前所未有的超大规模语言模型。

# ray + Megatron-DeepSpeed


- 将 Ray 与 Megatron-DeepSpeed 结合，通常不是用 Ray 直接启动 Megatron 的底层训练进程，而是利用 Ray 强大的分布式编排和调度能力来管理整个复杂的训练生命周期。

在这种架构中：
- Ray 扮演“总指挥”的角色，负责资源管理、任务调度、容错和状态监控。
- Megatron-DeepSpeed 扮演“执行者”的角色，专注于单机或多机多卡环境下的高性能模型训练。

这种模式在复杂的训练流程（如 RLHF）中尤其常见，其中 Ray 可以协调 Actor、Critic、Reference 和 Reward 等多个模型组件的协同工作。

下面是一个完整的代码示例，展示如何使用 Ray 来编排和启动一个 Megatron-DeepSpeed 训练任务。

## 🏗️ 架构设计：Ray 作为编排层
1. Ray Head Node: 负责整个集群的资源管理和任务调度。
2. Ray Worker Group: 由 Ray 创建并管理的一组工作进程（Ray Actor）。每个工作进程对应一个物理节点（或一个训练任务）。
3. Megatron-DeepSpeed 任务: 在每个 Ray Worker 内部，通过子进程的方式启动真正的 torchrun 训练脚本。

## 💻 代码实现
我们将代码分为三个部分：训练脚本、Ray 编排脚本和启动脚本。
1. 训练脚本 (train.py)
这个脚本与之前介绍的 Megatron-DeepSpeed 脚本基本相同，但现在是作为 Ray Worker 的子进程被启动的。

In [ ]:
# train.py
import torch
import deepspeed
import argparse
from megatron import get_args, initialize_megatron
from megatron.model import GPTModel
# ... 其他 Megatron 导入 ...

def model_provider(pre_process=True, post_process=True):
    # ... 模型定义 ...
    model = GPTModel(num_layers=12, hidden_size=768, num_attention_heads=12)
    return model

def add_model_config_args(parser):
    # ... 参数定义 ...
    return parser

def main():
    # 1. 初始化 Megatron
    initialize_megatron(extra_args_provider=add_model_config_args)
    args = get_args()

    # 2. 构建模型
    model = model_provider()

    # 3. DeepSpeed 配置
    ds_config = {
        "train_batch_size": args.global_batch_size,
        "zero_optimization": {
            "stage": 2,
            "overlap_comm": True,
            "contiguous_gradients": True,
        },
        "fp16": {"enabled": True},
        "optimizer": {"type": "Adam", "params": {"lr": args.lr}}
    }

    # 4. 初始化 DeepSpeed 引擎
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model, config=ds_config, dist_init_required=False
    )

    # 5. 训练循环
    for step in range(100):
        # ... 前向、反向、更新 ...
        # data = ...
        # loss = ...
        # model_engine.backward(loss)
        # model_engine.step()
        if step % 10 == 0:
            print(f"Rank {torch.distributed.get_rank()}: Step {step}, Loss: ...")

if __name__ == "__main__":
    main()

2. Ray 编排脚本 (ray_orchestrator.py)
- 这是核心部分，展示了如何使用 Ray Actor 来代表一个训练节点，并在其内部启动训练任务。

In [ ]:
import ray
import os
import subprocess
import time
from typing import List

# --- 1. 定义 Ray Worker ---
@ray.remote(num_gpus=8) # 假设每个节点有8个GPU
class MegatronWorker:
    def __init__(self, rank: int, world_size: int, master_addr: str, master_port: str):
        self.rank = rank
        self.world_size = world_size
        self.master_addr = master_addr
        self.master_port = master_port
        self.process = None

    def start_training(self, script_path: str, megatron_args: List[str]):
        """在当前 Worker 所在的节点上启动 Megatron-DeepSpeed 训练"""
        # 构建 torchrun 命令
        # 注意：这里我们是在 Ray Actor 内部启动一个新的分布式进程组
        cmd = [
            "torchrun",
            f"--nnodes={self.world_size}",
            f"--node_rank={self.rank}",
            f"--master_addr={self.master_addr}",
            f"--master_port={self.master_port}",
            f"--nproc_per_node=8", # 每个节点使用8个GPU
            script_path
        ] + megatron_args

        print(f"Rank {self.rank}: 正在启动训练任务: {' '.join(cmd)}")
        
        # 使用 subprocess 启动训练脚本
        # 这样可以解耦 Ray 进程和训练进程，便于独立管理
        self.process = subprocess.Popen(cmd)

    def stop_training(self):
        """停止训练任务"""
        if self.process:
            print(f"Rank {self.rank}: 正在停止训练任务...")
            self.process.terminate()
            self.process.wait()
            print(f"Rank {self.rank}: 训练任务已停止。")

    def is_alive(self):
        """检查训练任务是否仍在运行"""
        return self.process is not None and self.process.poll() is None

# --- 2. 定义 Ray Driver (主程序) ---
def main():
    # 1. 初始化 Ray 集群
    # 在实际生产中，这里会连接到已有的 Ray 集群
    if not ray.is_initialized():
        ray.init(address="auto") # 或 ray.init() 在本地启动

    # 2. 配置训练参数
    num_nodes = 2
    master_addr = "192.168.1.100" # Ray Head 节点的IP
    master_port = "6000"
    training_script = "train.py"
    
    megatron_args = [
        "--num-layers", "24",
        "--hidden-size", "1024",
        "--num-attention-heads", "16",
        "--global-batch-size", "1024",
        "--lr", "1e-4"
    ]

    # 3. 创建 Ray Worker Group
    # 使用 Ray 的 Placement Group 可以确保每个 Worker 被调度到不同的物理节点上
    # 这里为了简化，我们直接创建 Actor
    workers = []
    for i in range(num_nodes):
        # 创建 Actor 时，Ray 会自动分配资源（例如8个GPU）
        worker = MegatronWorker.remote(i, num_nodes, master_addr, master_port)
        workers.append(worker)
    
    print(f"已创建 {num_nodes} 个 Megatron Worker。")

    # 4. 启动所有 Worker 的训练任务
    # 使用 ray.get() 并行触发所有 Worker 的 start_training 方法
    # 这是一个异步操作，所有 Worker 会几乎同时开始训练
    print("正在启动所有训练任务...")
    ray.get([worker.start_training.remote(training_script, megatron_args) for worker in workers])
    print("所有训练任务已启动！")

    # 5. 监控训练状态
    try:
        while True:
            # 检查所有 Worker 的训练进程是否仍在运行
            statuses = ray.get([worker.is_alive.remote() for worker in workers])
            if not all(statuses):
                print("某个训练任务已退出，正在停止所有任务...")
                ray.get([worker.stop_training.remote() for worker in workers])
                break
            time.sleep(10) # 每10秒检查一次
    except KeyboardInterrupt:
        print("\n收到中断信号，正在优雅地停止所有任务...")
        ray.get([worker.stop_training.remote() for worker in workers])
    
    # 6. 清理
    ray.shutdown()

if __name__ == "__main__":
    main()

3. 启动脚本
- 你需要先启动一个 Ray 集群，然后运行编排脚本。

In [ ]:
# 1. 在 Head 节点上启动 Ray
ray start --head --port=6378

# 2. 在其他 Worker 节点上启动 Ray (连接到 Head 节点)
# ray start --address=<HEAD_NODE_IP>:6378

# 3. 在 Head 节点上运行 Ray 编排脚本
python ray_orchestrator.py

## 🔑 关键解释
- 分层架构:
    - Ray 层: ray_orchestrator.py 是顶层控制器。它不关心训练的具体细节，只负责“启动”、“停止”和“监控”任务。它通过创建 MegatronWorker 的 Ray Actor 来代表每个计算节点。
    - Megatron-DeepSpeed 层: train.py 是底层的执行单元。它在每个 Ray Worker 内部作为一个独立的子进程运行，专注于高性能的分布式训练计算。
- 资源调度:
    - @ray.remote(num_gpus=8) 告诉 Ray 每个 MegatronWorker 需要 8 个 GPU。Ray 的调度器会自动将这些 Actor 放置到满足资源要求的物理节点上，这比手动编写 hostfile 要灵活和强大得多。
- 生命周期管理:
    - Ray 提供了统一的接口来管理整个训练作业的生命周期。你可以轻松地在代码中实现优雅停机、故障恢复（例如，检测到某个 Worker 失败后重启它）和日志收集。
- 适用场景:
    - 这种模式在RLHF (Reinforcement Learning from Human Feedback) 等复杂场景中是标准做法。例如，OpenRLHF 等框架就使用 Ray 来协调 Actor、Critic、Reward 等多个模型组件的协同训练，每个组件本身可能就是一个 Megatron-DeepSpeed 任务。Ray 负责处理这些组件之间复杂的数据流和依赖关系，而 Megatron-DeepSpeed 则负责每个组件内部的高效训练。